Q777: Clinic_Wait_Time

In [8]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)
from pyspark.sql.functions import (
    col,
    to_timestamp,
    unix_timestamp,
    floor,
    sum,
    avg,
    count,
    lit
)


# ============================================================
# 1. Load Appointment Data
# ============================================================

def load_appointment_data(
    spark: SparkSession,
    file_path: str
) -> DataFrame:

    schema = StructType([
        StructField("appointment_id", StringType(), True),
        StructField("patient_id", StringType(), True),
        StructField("doctor", StringType(), True),
        StructField("department", StringType(), True),
        StructField("scheduled_time", StringType(), True),
        StructField("actual_time", StringType(), True),
        StructField("wait_reason", StringType(), True)
    ])

    return spark.read.csv(
        file_path,
        header=True,
        schema=schema
    )


# ============================================================
# 2. Append Wait Minutes
# ============================================================

def append_wait_minutes(
    df: DataFrame
) -> DataFrame:

    scheduled = unix_timestamp(
        to_timestamp(
            col("scheduled_time"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )

    actual = unix_timestamp(
        to_timestamp(
            col("actual_time"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )

    return df.withColumn(
        "wait_minutes",
        floor((actual - scheduled) / 60).cast("int")
    )


# ============================================================
# 3. Get Long Wait Appointments
# ============================================================

def get_long_wait_appointments(
    df: DataFrame,
    threshold_minutes: int
) -> DataFrame:

    df = append_wait_minutes(df)

    return df.filter(
        col("wait_minutes") > threshold_minutes
    )


# ============================================================
# 4. Most Delayed Doctor
# ============================================================

def most_delayed_doctor(
    df_with_wait: DataFrame
) -> DataFrame:

    return (
        df_with_wait
        .groupBy("doctor")
        .agg(
            sum("wait_minutes").alias("total_wait")
        )
        .orderBy(
            col("total_wait").desc(),
            col("doctor").asc()
        )
        .limit(1)
    )


# ============================================================
# 5. Long Wait Percentage
# ============================================================

def long_wait_percentage(
    df_with_wait: DataFrame
) -> float:

    total_count = df_with_wait.count()

    if total_count == 0:
        return 0.0

    long_wait_count = df_with_wait.filter(
        col("wait_minutes") > 30
    ).count()

    return float(
        (long_wait_count / total_count) * 100
    )


# ============================================================
# 6. Most Delayed Appointment
# ============================================================

def most_delayed_appointment(
    df_with_wait: DataFrame
) -> tuple[str, int]:

    result = (
        df_with_wait
        .orderBy(
            col("wait_minutes").desc(),
            col("appointment_id").asc()
        )
        .first()
    )

    if result is None:
        return ("", 0)

    return (
        result["appointment_id"],
        int(result["wait_minutes"])
    )


# ============================================================
# 7. Average Wait By Department
# ============================================================

def avg_wait_by_department(
    df_with_wait: DataFrame
) -> DataFrame:

    return (
        df_with_wait
        .groupBy("department")
        .agg(
            avg("wait_minutes").alias(
                "avg_wait_minutes"
            )
        )
        .orderBy(
            col("avg_wait_minutes").desc(),
            col("department").asc()
        )
    )


# ============================================================
# 8. Top N Patients By Wait
# ============================================================

def top_n_patients_by_wait(
    df_with_wait: DataFrame,
    n: int
) -> DataFrame:

    return (
        df_with_wait
        .groupBy("patient_id")
        .agg(
            sum("wait_minutes").alias(
                "total_wait_minutes"
            )
        )
        .orderBy(
            col("total_wait_minutes").desc(),
            col("patient_id").asc()
        )
        .limit(n)
    )


# ============================================================
# 9. Wait Reason Counts
# ============================================================

def wait_reason_counts(
    df: DataFrame
) -> DataFrame:

    return (
        df.groupBy("wait_reason")
        .agg(
            count("*").alias("reason_count")
        )
        .orderBy(
            col("reason_count").desc(),
            col("wait_reason").asc()
        )
        .select(
            "wait_reason",
            "reason_count"
        )
    )
spark = SparkSession.builder \
    .appName("Clinic Appointment Wait-Time") \
    .getOrCreate()

# 1. Load
df = load_appointment_data(spark, '/content/appointments.csv')

# 2. Add wait minutes
df_with_wait = append_wait_minutes(df)

print("=== Appointments With Wait ===")
df_with_wait.show()

# 3. Long waits > 30 minutes
print("=== Long Wait Appointments ===")
get_long_wait_appointments(
    df,
    30
).show()

# 4. Most delayed doctor
print("=== Most Delayed Doctor ===")
most_delayed_doctor(
    df_with_wait
).show()

# 5. Long wait percentage
print("=== Long Wait Percentage ===")
print(
    long_wait_percentage(
        df_with_wait
    )
)

# 6. Most delayed appointment
print("=== Most Delayed Appointment ===")
print(
    most_delayed_appointment(
        df_with_wait
    )
)

# 7. Average wait by department
print("=== Average Wait By Department ===")
avg_wait_by_department(
    df_with_wait
).show()

# 8. Top 3 patients
print("=== Top 3 Patients ===")
top_n_patients_by_wait(
    df_with_wait,
    3
).show()

# 9. Wait reason counts
print("=== Wait Reason Counts ===")
wait_reason_counts(
    df
).show()

spark.stop()

=== Appointments With Wait ===
+--------------+----------+--------+-----------+-------------------+-------------------+--------------+------------+
|appointment_id|patient_id|  doctor| department|     scheduled_time|        actual_time|   wait_reason|wait_minutes|
+--------------+----------+--------+-----------+-------------------+-------------------+--------------+------------+
|         A1001|       P01| Dr. Rao| Cardiology|2025-01-01 09:00:00|2025-01-01 09:12:00|  Late arrival|          12|
|         A1002|       P02| Dr. Rao| Cardiology|2025-01-01 10:00:00|2025-01-01 10:55:00|    Overbooked|          55|
|         A1003|       P03| Dr. Sen|Dermatology|2025-01-01 09:30:00|2025-01-01 09:50:00|  System delay|          20|
|         A1004|       P01| Dr. Sen|Dermatology|2025-01-02 11:00:00|2025-01-02 11:05:00|  Late arrival|           5|
|         A1005|       P04|Dr. Iyer|Orthopedics|2025-01-03 08:00:00|2025-01-03 09:10:00|Emergency case|          70|
+--------------+----------+------

Q1000: SmartCity Mobility Mega Assessment

In [10]:
from __future__ import annotations

from typing import List, Tuple

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    TimestampType,
    DateType,
    LongType,
)


# ============================================================
# A) Core Trip Analytics
# ============================================================

# 1. Define Trip Schema
def define_trip_schema() -> StructType:

    return StructType([
        StructField("trip_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("vehicle_id", StringType(), True),
        StructField("zone", StringType(), True),
        StructField("trip_date", StringType(), True),
        StructField("start_ts", StringType(), True),
        StructField("end_ts", StringType(), True),
        StructField("distance_km", DoubleType(), True),
        StructField("fare", DoubleType(), True),
        StructField("discount", DoubleType(), True),
        StructField("payment_method", StringType(), True),
        StructField("vehicle_type", StringType(), True),
        StructField("speed_kmh", DoubleType(), True),
        StructField("harsh_brake_count", IntegerType(), True),
        StructField("status", StringType(), True)
    ])


# 2. Load Trips
def load_trips(
    spark: SparkSession,
    path: str,
    schema: StructType
) -> DataFrame:

    return spark.read.csv(
        path,
        header=True,
        schema=schema
    )


# 3. Parse Trip Date
def parse_trip_date(
    df: DataFrame
) -> DataFrame:

    return df.withColumn(
        "trip_date",
        F.to_date(
            F.col("trip_date"),
            "yyyy-MM-dd"
        )
    )


# 4. Add Trip Duration
def add_trip_duration_min(
    df: DataFrame
) -> DataFrame:

    start = F.unix_timestamp(
        F.to_timestamp(
            F.col("start_ts"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )

    end = F.unix_timestamp(
        F.to_timestamp(
            F.col("end_ts"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )

    return df.withColumn(
        "trip_duration_min",
        F.round(
            (end - start) / 60
        ).cast("int")
    )


# 5. Add Cost Per KM
def add_cost_per_km(
    df: DataFrame
) -> DataFrame:

    return df.withColumn(
        "cost_per_km",
        (
            F.col("fare") /
            F.col("distance_km")
        ).cast("double")
    )


# 6. Filter Peak Hour Trips
def filter_peak_hour_trips(
    df: DataFrame
) -> DataFrame:

    hour = F.hour(
        F.to_timestamp(
            F.col("start_ts"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )

    return df.filter(
        (
            (hour >= 7) &
            (hour <= 10)
        ) |
        (
            (hour >= 17) &
            (hour <= 20)
        )
    )


# 7. Top N Users By Distance
def top_n_users_by_distance(
    df: DataFrame,
    n: int
) -> DataFrame:

    return (
        df.groupBy("user_id")
        .agg(
            F.sum("distance_km").alias(
                "total_distance_km"
            )
        )
        .orderBy(
            F.col("total_distance_km").desc(),
            F.col("user_id").asc()
        )
        .limit(n)
    )


# 8. Average Speed By Zone
def avg_speed_by_zone(
    df: DataFrame
) -> DataFrame:

    return (
        df.groupBy("zone")
        .agg(
            F.avg("speed_kmh").alias(
                "avg_speed_kmh"
            )
        )
    )


# 9. Most Common Vehicle Type
def most_common_vehicle_type(
    df: DataFrame
) -> str:

    result = (
        df.groupBy("vehicle_type")
        .count()
        .orderBy(
            F.col("count").desc(),
            F.col("vehicle_type").asc()
        )
        .first()
    )

    if result is None:
        return ""

    return str(result["vehicle_type"])


# 10. Cancellation Rate By Zone
def cancellation_rate_by_zone(
    df: DataFrame
) -> DataFrame:

    result = (
        df.groupBy("zone")
        .agg(
            F.count("*").alias("total"),
            F.sum(
                F.when(
                    F.col("status") == "Cancelled",
                    1
                ).otherwise(0)
            ).alias("cancelled")
        )
        .withColumn(
            "cancellation_rate",
            (
                F.col("cancelled") /
                F.col("total")
            ).cast("double")
        )
    )

    return result.select(
        "zone",
        "cancellation_rate"
    )


# 11. High Duration Trips
def high_duration_trips(
    df: DataFrame,
    threshold_min: int
) -> DataFrame:

    if "trip_duration_min" not in df.columns:
        df = add_trip_duration_min(df)

    return df.filter(
        F.col("trip_duration_min") >
        threshold_min
    )


# 12. Count Active Vehicles By Zone
def count_active_vehicles_by_zone(
    vehicles_df: DataFrame
) -> DataFrame:

    return (
        vehicles_df
        .filter(
            F.col("status") == "Active"
        )
        .groupBy("home_zone")
        .agg(
            F.countDistinct("vehicle_id").alias(
                "active_vehicles"
            )
        )
        .withColumnRenamed(
            "home_zone",
            "zone"
        )
    )


# 13. Daily Net Revenue Trend
def daily_net_revenue_trend(
    df: DataFrame
) -> DataFrame:

    return (
        df.withColumn(
            "net_revenue",
            (
                F.col("fare") -
                F.col("discount")
            ).cast("double")
        )
        .groupBy("trip_date")
        .agg(
            F.sum("net_revenue").alias(
                "daily_net_revenue"
            )
        )
        .orderBy(
            F.col("trip_date").asc()
        )
    )


# 14. Top Vehicle By Net Revenue
def top_vehicle_by_net_revenue(
    df: DataFrame
) -> Tuple[str, float]:

    result = (
        df.withColumn(
            "net_revenue",
            F.col("fare") -
            F.col("discount")
        )
        .groupBy("vehicle_id")
        .agg(
            F.sum("net_revenue").alias(
                "total_net_revenue"
            )
        )
        .orderBy(
            F.col("total_net_revenue").desc(),
            F.col("vehicle_id").asc()
        )
        .first()
    )

    if result is None:
        return ("", 0.0)

    return (
        str(result["vehicle_id"]),
        float(result["total_net_revenue"])
    )


# 15. List Zones
def list_zones(
    df: DataFrame
) -> List[str]:

    rows = (
        df.select("zone")
        .distinct()
        .collect()
    )

    return sorted(
        [str(row["zone"]) for row in rows]
    )


# 16. Trips In Date Range
def trips_in_date_range(
    df: DataFrame,
    start: str,
    end: str
) -> DataFrame:

    return df.filter(
        F.to_date(
            F.col("trip_date")
        ).between(
            F.to_date(F.lit(start)),
            F.to_date(F.lit(end))
        )
    )


# 17. Flag Safety Risk
def flag_safety_risk(
    df: DataFrame,
    speed_threshold: float,
    brake_threshold: int
) -> DataFrame:

    return df.withColumn(
        "is_risky",
        (
            (F.col("speed_kmh") > speed_threshold) |
            (
                F.col("harsh_brake_count") >
                brake_threshold
            )
        ).cast("boolean")
    )


# 18. Top N Risky Vehicles
def top_n_risky_vehicles(
    df: DataFrame,
    n: int
) -> DataFrame:

    return (
        df.filter(
            F.col("is_risky") == True
        )
        .groupBy("vehicle_id")
        .agg(
            F.count("*").alias(
                "risky_count"
            )
        )
        .orderBy(
            F.col("risky_count").desc(),
            F.col("vehicle_id").asc()
        )
        .limit(n)
    )


# 19. Average Fare By Payment Method
def avg_fare_by_payment_method(
    df: DataFrame
) -> DataFrame:

    return (
        df.groupBy("payment_method")
        .agg(
            F.avg("fare").alias(
                "avg_fare"
            )
        )
    )


# 20. Get Longest Trip
def get_longest_trip(
    df: DataFrame
) -> Tuple[str, int]:

    if "trip_duration_min" not in df.columns:
        df = add_trip_duration_min(df)

    result = (
        df.orderBy(
            F.col("trip_duration_min").desc(),
            F.col("trip_id").asc()
        )
        .first()
    )

    if result is None:
        return ("", 0)

    return (
        str(result["trip_id"]),
        int(result["trip_duration_min"])
    )


# ============================================================
# B) Inline Join + Enrichment
# ============================================================

# 21. User Profile Schema
def define_user_profile_schema() -> StructType:

    return StructType([
        StructField("user_id", StringType(), True),
        StructField("first_name", StringType(), True),
        StructField("last_name", StringType(), True),
        StructField("full_name", StringType(), True),
        StructField("plan_id", StringType(), True)
    ])


# 22. Vehicle / Plan Schema
def define_vehicle_schema() -> StructType:

    return StructType([
        StructField("plan_id", StringType(), True),
        StructField("plan_name", StringType(), True)
    ])


# 23. Load Inline Profiles And Plans
def load_inline_profiles_and_plans(
    spark: SparkSession,
    profile_schema: StructType,
    plan_schema: StructType
) -> Tuple[DataFrame, DataFrame]:

    profiles_rows = [
        ("U1", "Aarav", "Sharma", None, "P2"),
        ("U2", "Diya", "Iyer", "Diya Iyer", "P1"),
        ("U3", "Kabir", "Singh", None, "P1"),
        ("U4", "Meera", "Nair", None, "P2"),
    ]

    plans_rows = [
        ("P1", "Standard"),
        ("P2", "Premium"),
    ]

    profiles_df = spark.createDataFrame(
        profiles_rows,
        schema=profile_schema
    )

    plans_df = spark.createDataFrame(
        plans_rows,
        schema=plan_schema
    )

    return profiles_df, plans_df


# 24. Join Profiles With Plans
def join_profiles_with_plans(
    profiles_df: DataFrame,
    plans_df: DataFrame
) -> DataFrame:

    return (
        profiles_df
        .join(
            plans_df,
            on="plan_id",
            how="left"
        )
        .select(
            "user_id",
            "first_name",
            "last_name",
            "full_name",
            "plan_id",
            "plan_name"
        )
    )


# 25. Enrich Full Name
def enrich_full_name(
    profile_joined_df: DataFrame
) -> DataFrame:

    full_name = F.when(
        F.col("full_name").isNull() |
        (
            F.length(
                F.trim(
                    F.col("full_name")
                )
            ) == 0
        ),
        F.concat_ws(
            " ",
            F.col("first_name"),
            F.col("last_name")
        )
    ).otherwise(
        F.col("full_name")
    )

    return profile_joined_df.withColumn(
        "full_name",
        full_name
    )


# ============================================================
# C) Unix Timestamp Conversions
# ============================================================

# 26. Add Start Epoch Seconds
def add_start_epoch_seconds(
    df: DataFrame
) -> DataFrame:

    return df.withColumn(
        "start_epoch_seconds",
        F.unix_timestamp(
            F.to_timestamp(
                F.col("start_ts"),
                "yyyy-MM-dd HH:mm:ss"
            )
        ).cast("long")
    )


# 27. Add Start Timestamp From Epoch
def add_start_ts_from_epoch(
    df: DataFrame
) -> DataFrame:

    return df.withColumn(
        "start_ts_from_epoch",
        F.from_unixtime(
            F.col("start_epoch_seconds")
        ).cast("timestamp")
    )


# ============================================================
# D) Additional Analytics
# ============================================================

# 28. Revenue Share By Vehicle Type
def revenue_share_by_vehicle_type(
    df: DataFrame
) -> List[str]:

    grouped = (
        df.groupBy("vehicle_type")
        .agg(
            F.sum("rev").alias("total_rev")
        )
    )

    grand_total = grouped.agg(
        F.sum("total_rev").alias("grand_total")
    ).first()["grand_total"]

    if grand_total is None or float(grand_total) == 0:
        return []

    result = (
        grouped
        .withColumn(
            "share",
            F.col("total_rev") /
            F.lit(float(grand_total))
        )
        .orderBy(
            F.col("share").desc(),
            F.col("vehicle_type").asc()
        )
        .select("vehicle_type")
        .collect()
    )

    return [
        str(row["vehicle_type"])
        for row in result
    ]


# 29. Busiest Zone By Trips
def busiest_zone_by_trips(
    df: DataFrame
) -> Tuple[str, int]:

    result = (
        df.groupBy("zone")
        .count()
        .orderBy(
            F.col("count").desc(),
            F.col("zone").asc()
        )
        .first()
    )

    if result is None:
        return ("", 0)

    return (
        str(result["zone"]),
        int(result["count"])
    )


# 30. Completed Trips By Zone
def completed_trips_by_zone(
    df: DataFrame
) -> DataFrame:

    return (
        df.filter(
            F.col("status") == "Completed"
        )
        .groupBy("zone")
        .agg(
            F.count("*").alias(
                "completed_trips"
            )
        )
        .orderBy(
            F.col("completed_trips").desc(),
            F.col("zone").asc()
        )
    )


# 31. Add Utilization Score
def add_utilization_score(
    df: DataFrame
) -> DataFrame:

    if "trip_duration_min" not in df.columns:
        df = add_trip_duration_min(df)

    return df.withColumn(
        "utilization_score",
        (
            F.col("distance_km") *
            F.col("trip_duration_min")
        ).cast("double")
    )


# 32. Top N Users By Utilization
def top_n_users_by_utilization(
    df: DataFrame,
    n: int
) -> DataFrame:

    if "utilization_score" not in df.columns:
        df = add_utilization_score(df)

    return (
        df.groupBy("user_id")
        .agg(
            F.sum("utilization_score").alias(
                "total_utilization"
            )
        )
        .orderBy(
            F.col("total_utilization").desc(),
            F.col("user_id").asc()
        )
        .limit(n)
    )


# 33. Median Duration By Zone
def median_duration_by_zone(
    df: DataFrame
) -> DataFrame:

    if "trip_duration_min" not in df.columns:
        df = add_trip_duration_min(df)

    return (
        df.groupBy("zone")
        .agg(
            F.percentile_approx(
                "trip_duration_min",
                0.5
            ).alias(
                "median_duration_min"
            )
        )
    )


# 34. Min-Max Normalize Fare
def minmax_normalize_fare(
    df: DataFrame
) -> DataFrame:

    stats = df.agg(
        F.min("fare").alias("min_fare"),
        F.max("fare").alias("max_fare")
    ).first()

    min_fare = stats["min_fare"]
    max_fare = stats["max_fare"]

    if (
        min_fare is None or
        max_fare is None or
        float(max_fare) == float(min_fare)
    ):
        return df.withColumn(
            "fare_norm",
            F.lit(0.0)
        )

    return df.withColumn(
        "fare_norm",
        (
            (F.col("fare") - F.lit(float(min_fare))) /
            (
                F.lit(float(max_fare)) -
                F.lit(float(min_fare))
            )
        ).cast("double")
    )


# 35. Detect Outlier Fares
def detect_outlier_fares(
    df: DataFrame
) -> DataFrame:

    stats = df.agg(
        F.avg("fare").alias("mean_fare"),
        F.stddev_pop("fare").alias("std_fare")
    ).first()

    mean_fare = stats["mean_fare"]
    std_fare = stats["std_fare"]

    if mean_fare is None or std_fare is None:
        return df.withColumn(
            "is_outlier",
            F.lit(False)
        )

    threshold = (
        float(mean_fare) +
        2 * float(std_fare)
    )

    return df.withColumn(
        "is_outlier",
        F.col("fare") > F.lit(threshold)
    )


# 36. Monthly Net Revenue By Zone
def monthly_net_revenue_by_zone(
    df: DataFrame
) -> DataFrame:

    return (
        df.withColumn(
            "month",
            F.date_format(
                F.to_date(
                    F.col("trip_date")
                ),
                "yyyy-MM"
            )
        )
        .withColumn(
            "net_revenue",
            (
                F.col("fare") -
                F.col("discount")
            ).cast("double")
        )
        .groupBy(
            "month",
            "zone"
        )
        .agg(
            F.sum("net_revenue").alias(
                "monthly_net_revenue"
            )
        )
        .orderBy(
            F.col("month").asc(),
            F.col("zone").asc()
        )
    )


# 37. Weekday Peak Trip Counts
def weekday_peak_trip_counts(
    df: DataFrame
) -> DataFrame:

    peak_df = filter_peak_hour_trips(df)

    return (
        peak_df.withColumn(
            "weekday",
            F.date_format(
                F.to_date(
                    F.col("trip_date")
                ),
                "E"
            )
        )
        .groupBy("weekday")
        .agg(
            F.count("*").alias(
                "peak_trips"
            )
        )
    )


# 38. Pivot Payment Counts By Zone
def pivot_payment_counts_by_zone(
    df: DataFrame
) -> DataFrame:

    return (
        df.groupBy("zone")
        .pivot("payment_method")
        .count()
        .fillna(0)
    )


# 39. Calculate Net Revenue
def calculate_net_revenue(
    df: DataFrame
) -> DataFrame:

    return df.withColumn(
        "net_revenue",
        (
            F.col("fare") -
            F.col("discount")
        ).cast("double")
    )


# 40. Flag Service Due
def flag_service_due(
    df: DataFrame,
    due_days: int
) -> DataFrame:

    service_date = F.to_date(
        F.col("last_service_date")
    )

    days_since_service = F.datediff(
        F.current_date(),
        service_date
    )

    return df.withColumn(
        "is_service_due",
        days_since_service > due_days
    )

from pyspark.sql import SparkSession


if __name__ == "__main__":

    # 1. Create SparkSession
    spark = (
        SparkSession.builder
        .appName("SmartCity Mobility Mega Assessment")
        .getOrCreate()
    )

    # --------------------------------------------------
    # LOAD TRIP DATA
    # --------------------------------------------------

    trip_schema = define_trip_schema()

    df = load_trips(
        spark,
        "/content/mobility_trips.csv", # Corrected path
        trip_schema
    )

    print("Original Data")
    df.show()

    # --------------------------------------------------
    # 1 - 20 : CORE TRIP ANALYTICS
    # --------------------------------------------------

    df = parse_trip_date(df)
    df = add_trip_duration_min(df)
    df = add_cost_per_km(df)

    print("Peak Hour Trips")
    filter_peak_hour_trips(df).show()

    print("Top Users by Distance")
    top_n_users_by_distance(df, 5).show()

    print("Average Speed by Zone")
    avg_speed_by_zone(df).show()

    print("Most Common Vehicle Type")
    print(most_common_vehicle_type(df))

    print("Cancellation Rate by Zone")
    cancellation_rate_by_zone(df).show()

    print("High Duration Trips")
    high_duration_trips(df, 30).show()

    # --------------------------------------------------
    # VEHICLE DATA
    # --------------------------------------------------

    # Example:
    # vehicles_df = spark.read.csv(
    #     "data/vehicles.csv",
    #     header=True,
    #     inferSchema=True
    # )
    #
    # print("Active Vehicles by Zone")
    # count_active_vehicles_by_zone(vehicles_df).show()

    print("Daily Net Revenue")
    daily_net_revenue_trend(df).show()

    print("Top Vehicle by Net Revenue")
    print(top_vehicle_by_net_revenue(df))

    print("Zones")
    print(list_zones(df))

    print("Trips in Date Range")
    trips_in_date_range(
        df,
        "2026-01-01",
        "2026-12-31"
    ).show()

    print("Safety Risk")
    risky_df = flag_safety_risk(
        df,
        speed_threshold=80,
        brake_threshold=3
    )
    risky_df.show()

    print("Top Risky Vehicles")
    top_n_risky_vehicles(risky_df, 5).show()

    print("Average Fare by Payment Method")
    avg_fare_by_payment_method(df).show()

    print("Longest Trip")
    print(get_longest_trip(df))

    # --------------------------------------------------
    # 21 - 25 : PROFILES + PLANS
    # --------------------------------------------------

    profile_schema = define_user_profile_schema()
    plan_schema = define_vehicle_schema()

    profiles_df, plans_df = load_inline_profiles_and_plans(
        spark,
        profile_schema,
        plan_schema
    )

    print("Profiles")
    profiles_df.show()

    print("Plans")
    plans_df.show()

    profile_joined = join_profiles_with_plans(
        profiles_df,
        plans_df
    )

    print("Joined Profiles + Plans")
    profile_joined.show()

    enriched_profiles = enrich_full_name(
        profile_joined
    )

    print("Enriched Profiles")
    enriched_profiles.show()

    # --------------------------------------------------
    # 26 - 27 : EPOCH
    # --------------------------------------------------

    epoch_df = add_start_epoch_seconds(df)

    print("Start Epoch Seconds")
    epoch_df.show()

    restored_df = add_start_ts_from_epoch(epoch_df)

    print("Restored Timestamp")
    restored_df.show()

    # --------------------------------------------------
    # 28 - 40 : ADDITIONAL ANALYTICS
    # --------------------------------------------------

    # Function 28 expects column: rev
    print("Revenue Share by Vehicle Type")

    if "rev" in df.columns:
        print(
            revenue_share_by_vehicle_type(df)
        )

    print("Busiest Zone")
    print(busiest_zone_by_trips(df))

    print("Completed Trips by Zone")
    completed_trips_by_zone(df).show()

    utilization_df = add_utilization_score(df)

    print("Utilization Score")
    utilization_df.show()

    print("Top Users by Utilization")
    top_n_users_by_utilization(
        utilization_df,
        5
    ).show()

    print("Median Duration by Zone")
    median_duration_by_zone(df).show()

    normalized_df = minmax_normalize_fare(df)

    print("Normalized Fare")
    normalized_df.show()

    outlier_df = detect_outlier_fares(df)

    print("Fare Outliers")
    outlier_df.show()

    print("Monthly Net Revenue by Zone")
    monthly_net_revenue_by_zone(df).show()

    print("Weekday Peak Trip Counts")
    weekday_peak_trip_counts(df).show()

    print("Payment Counts by Zone")
    pivot_payment_counts_by_zone(df).show()

    net_df = calculate_net_revenue(df)

    print("Net Revenue")
    net_df.show()

    # Function 40 needs last_service_date column.
    # If your vehicle/service dataset has this column:
    #
    # service_df = flag_service_due(
    #     service_df,
    #     due_days=180
    # )
    # service_df.show()

    # --------------------------------------------------
    # STOP SPARK
    # --------------------------------------------------

    spark.stop()


Original Data
+-------+-------+----------+-----+----------+-------------------+-------------------+-----------+-----+--------+--------------+------------+---------+-----------------+---------+
|trip_id|user_id|vehicle_id| zone| trip_date|           start_ts|             end_ts|distance_km| fare|discount|payment_method|vehicle_type|speed_kmh|harsh_brake_count|   status|
+-------+-------+----------+-----+----------+-------------------+-------------------+-----------+-----+--------+--------------+------------+---------+-----------------+---------+
|   T001|     U1|        V1|ZoneA|2025-12-20|2025-12-20 08:00:00|2025-12-20 08:20:00|        5.0| 50.0|     5.0|          Card|        Bike|     15.0|                0|Completed|
|   T002|     U2|        V2|ZoneB|2025-12-20|2025-12-20 18:00:00|2025-12-20 18:45:00|       12.0|120.0|     0.0|           UPI|         Car|     16.0|                1|Completed|
|   T003|     U1|        V3|ZoneA|2025-12-21|2025-12-21 09:30:00|2025-12-21 10:10:00|      